In [1]:
import numpy as np
import pandas as pd
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")

In [3]:
movies = movies[['title','overview','genres','keywords']]
movies.dropna(inplace=True)

In [4]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [5]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [6]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [7]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords']

In [10]:
new_df = movies[['title','tags']]

In [25]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = tfidf.fit_transform(new_df['tags'])

In [12]:
type(new_df['tags'].iloc[0])

list

In [16]:
new_df = movies[['title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [17]:
type(new_df['tags'].iloc[0])

str

In [18]:
vectors = tfidf.fit_transform(new_df['tags'])

In [19]:
vectors.shape

(4800, 5000)

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [21]:
similarity.shape

(4800, 4800)

In [22]:
def recommend(movie):

    movie = movie.lower()

    if movie not in new_df['title'].str.lower().values:
        print("Movie not found!")
        return

    index = new_df[new_df['title'].str.lower() == movie].index[0]

    distances = sorted(
        list(enumerate(similarity[index])),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nTop 5 Movies Similar to '{movie.title()}':\n")

    for i in distances[1:6]:
        print(new_df.iloc[i[0]]['title'])

In [28]:
recommend("Avatar")


Top 5 Movies Similar to 'Avatar':

Aliens
Silent Running
Alien³
Moonraker
Spaceballs
